In [1]:
import os
import sys
import pandas as pd
import numpy as np
import anndata as ad
import glob
import time
import scipy
import scipy.sparse as sp
from scipy.sparse import csr_matrix
import anndata as an
import scanpy as sc
from collections import Counter
import matplotlib.pyplot as plt
from matplotlib import colormaps
import seaborn as sns
import matplotlib.patches as mpatches
import networkx as nx
import random
from importlib import reload
import warnings
import ot
from scipy.spatial.distance import pdist, squareform
from scipy.sparse import issparse
from matplotlib.colors import ListedColormap
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
from matplotlib.collections import LineCollection
from itertools import chain
import re
import pickle as pkl
from sklearn.decomposition import TruncatedSVD
import pyarrow
from sklearn.preprocessing import MinMaxScaler

"""WARNING: no warnings"""
warnings.filterwarnings("ignore")

source_path = os.path.abspath("../utilities/")
sys.path.append(source_path)
import matrix as mtrx
import utils as ut
#import plotting as plt2
source_path = os.path.abspath("../utilities/calculations/")
sys.path.append(source_path)
import centrality as central


/home/jduhamel/.conda/envs/pore_c/lib/python3.12/site-packages/scanpy/_utils/__init__.py:33: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  from anndata import __version__ as anndata_version
/home/jduhamel/.conda/envs/pore_c/lib/python3.12/site-packages/scanpy/__init__.py:24: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):
/home/jduhamel/.conda/envs/pore_c/lib/python3.12/site-packages/scanpy/readwrite.py:16: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):


In [2]:
%%time 
# 1MB resolution
resolution = 1000000

fpath = f"/nfs/turbo/umms-indikar/shared/projects/poreC/pipeline_outputs/higher_order/anndata/population_mESC_{resolution}_features.h5ad"

adata = sc.read_h5ad(fpath)

sc.logging.print_memory_usage()

Memory usage: current 2.79 GB, difference +2.79 GB
CPU times: user 12.2 s, sys: 2.77 s, total: 14.9 s
Wall time: 18.3 s


In [3]:
obs_df = pd.read_csv('/home/jduhamel/nodes_core_full.csv')
var_df = pd.read_csv('/home/jduhamel/edges_core_full.csv')

obs_df.set_index("bin_name", inplace=True)
var_df.set_index("read_name", inplace=True)


In [4]:
obs_df.head()

,bin_index,bin_start,bin_end,bin,chrom,chrom_bin,degree,genes,n_genes,ATACSeq_1,...,CTCF,H3K27ac,H3K27me3,RNA_1,RNA_2,RNA_3,RNA_4,RNA_5,RNA_6,PolII
bin_name,,,,,,,,,,,,,,,,,,,,,
chr9:121,0,121000000,122000000,121,chr9,121,3532,Gm47092;Higd1a;Gm47108;Gask1a;Lyzl4os;Gm47112;...,41,0.826484,...,1.149226,1.349552,0.866066,0.295370,0.387013,0.162437,0.225783,0.573875,0.122417,1.016765
chr19:26,1,26000000,27000000,26,chr19,26,3346,Gm50378;Smarca2;Gm50376;Gm48775;Gm815;1700048O...,11,0.497386,...,0.547185,0.336787,0.839273,0.133999,0.148066,0.132291,0.094680,0.141617,0.084908,0.417178
chr4:127,2,127000000,128000000,127,chr4,127,3768,Gjb3;Gm22221;Gm12943;Zmym6;Tmem35b;Gjb5;Gm1294...,16,0.754788,...,1.027046,1.577616,0.839461,0.521087,0.998980,0.293742,0.247401,0.906364,0.256514,0.750054
chr10:57,4,57000000,58000000,57,chr10,57,3285,Serinc1;Smpdl3a;Rpl48-ps1;Gm19256;Gm48055;Fabp...,14,0.472903,...,0.383735,0.229388,0.600799,0.238078,0.560188,0.233549,0.215554,0.490113,0.182086,0.176728
chr12:8,5,8000000,9000000,8,chr12,8,3557,Gm56512;Ldah;5033421B08Rik;Rhob;Gm33037;Laptm4...,25,0.690311,...,0.851758,0.774613,0.833768,0.229539,0.501868,0.283553,0.190379,0.617768,0.222526,0.741199


In [5]:
var_df.head()

,read_index,basename,mean_mapq,median_mapq,n_chromosomes,order,n_bins,read_length_bp,genes,n_genes
read_name,,,,,,,,,,
3f354c45-5e48-4f6d-8c7e-05369432b344,6,batch04,42.500000,55.0,2,4,2,5998,Gm56531,1
837b767f-c1b0-48b3-9221-e9f02cd9113b,256,batch01,60.000000,60.0,1,3,2,3076,Auts2,1
4f0afd95-f7e7-4d46-8304-15865e16ea76,279,batch04,31.600000,25.5,1,10,2,4107,Unc79,1
b06e4578-bbc1-4b7c-ae39-8202762b20b0,396,batch04,45.666667,60.0,2,18,2,5975,Ube4b;Auts2,2
4d1bca2e-b33c-4f4a-abec-702aaf308d99,447,batch04,60.000000,60.0,2,3,2,3733,4732471J01Rik,1


In [6]:
for col in obs_df.columns:
    print(f"{col}: {obs_df[col].dtype} | {type(obs_df[col].dtype)}")

bin_index: int64 | <class 'numpy.dtypes.Int64DType'>
bin_start: int64 | <class 'numpy.dtypes.Int64DType'>
bin_end: int64 | <class 'numpy.dtypes.Int64DType'>
bin: int64 | <class 'numpy.dtypes.Int64DType'>
chrom: object | <class 'numpy.dtypes.ObjectDType'>
chrom_bin: int64 | <class 'numpy.dtypes.Int64DType'>
degree: int64 | <class 'numpy.dtypes.Int64DType'>
genes: object | <class 'numpy.dtypes.ObjectDType'>
n_genes: int64 | <class 'numpy.dtypes.Int64DType'>
ATACSeq_1: float64 | <class 'numpy.dtypes.Float64DType'>
ATACSeq_2: float64 | <class 'numpy.dtypes.Float64DType'>
ATACSeq_3: float64 | <class 'numpy.dtypes.Float64DType'>
CTCF: float64 | <class 'numpy.dtypes.Float64DType'>
H3K27ac: float64 | <class 'numpy.dtypes.Float64DType'>
H3K27me3: float64 | <class 'numpy.dtypes.Float64DType'>
RNA_1: float64 | <class 'numpy.dtypes.Float64DType'>
RNA_2: float64 | <class 'numpy.dtypes.Float64DType'>
RNA_3: float64 | <class 'numpy.dtypes.Float64DType'>
RNA_4: float64 | <class 'numpy.dtypes.Float64DT

In [7]:
for col in var_df.columns:
    print(f"{col}: {var_df[col].dtype} | {type(var_df[col].dtype)}")

read_index: int64 | <class 'numpy.dtypes.Int64DType'>
basename: object | <class 'numpy.dtypes.ObjectDType'>
mean_mapq: float64 | <class 'numpy.dtypes.Float64DType'>
median_mapq: float64 | <class 'numpy.dtypes.Float64DType'>
n_chromosomes: int64 | <class 'numpy.dtypes.Int64DType'>
order: int64 | <class 'numpy.dtypes.Int64DType'>
n_bins: int64 | <class 'numpy.dtypes.Int64DType'>
read_length_bp: int64 | <class 'numpy.dtypes.Int64DType'>
genes: object | <class 'numpy.dtypes.ObjectDType'>
n_genes: int64 | <class 'numpy.dtypes.Int64DType'>


In [8]:
obs_df.index = pd.Index(obs_df.index.tolist(), dtype=object, name=obs_df.index.name)
var_df.index = pd.Index(var_df.index.tolist(), dtype=object, name=var_df.index.name)


for col in obs_df.columns:
    if isinstance(obs_df[col].array, pd.arrays.ArrowStringArray):
        obs_df[col] = pd.Series(
            obs_df[col].tolist(),      # force out of Arrow
            index=obs_df.index,
            dtype=object,             # force regular Python/object strings
            name=col
        )

for col in var_df.columns:
    if isinstance(var_df[col].array, pd.arrays.ArrowStringArray):
        var_df[col] = pd.Series(
            var_df[col].tolist(),      # force out of Arrow
            index=var_df.index,
            dtype=object,             # force regular Python/object strings
            name=col
        )

In [9]:
for col in obs_df.columns:
    print(f"{col}: {obs_df[col].dtype} | {type(obs_df[col].dtype)}")

bin_index: int64 | <class 'numpy.dtypes.Int64DType'>
bin_start: int64 | <class 'numpy.dtypes.Int64DType'>
bin_end: int64 | <class 'numpy.dtypes.Int64DType'>
bin: int64 | <class 'numpy.dtypes.Int64DType'>
chrom: object | <class 'numpy.dtypes.ObjectDType'>
chrom_bin: int64 | <class 'numpy.dtypes.Int64DType'>
degree: int64 | <class 'numpy.dtypes.Int64DType'>
genes: object | <class 'numpy.dtypes.ObjectDType'>
n_genes: int64 | <class 'numpy.dtypes.Int64DType'>
ATACSeq_1: float64 | <class 'numpy.dtypes.Float64DType'>
ATACSeq_2: float64 | <class 'numpy.dtypes.Float64DType'>
ATACSeq_3: float64 | <class 'numpy.dtypes.Float64DType'>
CTCF: float64 | <class 'numpy.dtypes.Float64DType'>
H3K27ac: float64 | <class 'numpy.dtypes.Float64DType'>
H3K27me3: float64 | <class 'numpy.dtypes.Float64DType'>
RNA_1: float64 | <class 'numpy.dtypes.Float64DType'>
RNA_2: float64 | <class 'numpy.dtypes.Float64DType'>
RNA_3: float64 | <class 'numpy.dtypes.Float64DType'>
RNA_4: float64 | <class 'numpy.dtypes.Float64DT

In [10]:
for col in var_df.columns:
    print(f"{col}: {var_df[col].dtype} | {type(var_df[col].dtype)}")

read_index: int64 | <class 'numpy.dtypes.Int64DType'>
basename: object | <class 'numpy.dtypes.ObjectDType'>
mean_mapq: float64 | <class 'numpy.dtypes.Float64DType'>
median_mapq: float64 | <class 'numpy.dtypes.Float64DType'>
n_chromosomes: int64 | <class 'numpy.dtypes.Int64DType'>
order: int64 | <class 'numpy.dtypes.Int64DType'>
n_bins: int64 | <class 'numpy.dtypes.Int64DType'>
read_length_bp: int64 | <class 'numpy.dtypes.Int64DType'>
genes: object | <class 'numpy.dtypes.ObjectDType'>
n_genes: int64 | <class 'numpy.dtypes.Int64DType'>


In [11]:
node_core = obs_df.index
edge_core = var_df.index

In [12]:
node_mask = adata.obs_names.isin(node_core)
edge_mask = adata.var_names.isin(edge_core)

adata_core = adata[node_mask, edge_mask].copy()
H_core = adata_core.X.tocsr().astype(float)

In [13]:
cdata = an.AnnData(X=H_core, obs=obs_df, var=var_df)

In [14]:
cdata.obs.index = pd.Index(cdata.obs.index.tolist(), dtype=object, name=cdata.obs.index.name)
cdata.var.index = pd.Index(cdata.var.index.tolist(), dtype=object, name=cdata.var.index.name)


for col in cdata.obs.columns:
    if isinstance(cdata.obs[col].array, pd.arrays.ArrowStringArray):
        cdata.obs[col] = pd.Series(
            cdata.obs[col].tolist(),      # force out of Arrow
            index=cdata.obs.index,
            dtype=object,             # force regular Python/object strings
            name=col
        )

for col in cdata.var.columns:
    if isinstance(cdata.var[col].array, pd.arrays.ArrowStringArray):
        cdata.var[col] = pd.Series(
            cdata.var[col].tolist(),      # force out of Arrow
            index=cdata.var.index,
            dtype=object,             # force regular Python/object strings
            name=col
        )

In [15]:
write_cdata=cdata.copy()

In [16]:
write_cdata.write_h5ad('/scratch/indikar_root/indikar1/jduhamel/pore_c/population_mESC_100000_core_current_1.h5ad')

In [17]:
adata_core.obs.index = pd.Index(adata_core.obs.index.tolist(), dtype=object, name=adata_core.obs.index.name)
adata_core.var.index = pd.Index(adata_core.var.index.tolist(), dtype=object, name=adata_core.var.index.name)


for col in adata_core.obs.columns:
    if isinstance(adata_core.obs[col].array, pd.arrays.ArrowStringArray):
        adata_core.obs[col] = pd.Series(
            adata_core.obs[col].tolist(),      # force out of Arrow
            index=adata_core.obs.index,
            dtype=object,             # force regular Python/object strings
            name=col
        )

for col in adata_core.var.columns:
    if isinstance(adata_core.var[col].array, pd.arrays.ArrowStringArray):
        adata_core.var[col] = pd.Series(
            adata_core.var[col].tolist(),      # force out of Arrow
            index=adata_core.var.index,
            dtype=object,             # force regular Python/object strings
            name=col
        )

In [18]:
write_adata_core=adata_core.copy()

In [19]:
write_adata_core.write_h5ad('/scratch/indikar_root/indikar1/jduhamel/pore_c/population_mESC_100000_core_current_2.h5ad')